# Article 1 — Notebook maître : benchmark complet, figures et tables du papier

**Papier :** *A Leakage-Audited Benchmark of Deep and Ensemble Detectors on the GeNIS 2025 Corpus*.

Ce notebook exécute **tout le plan expérimental** et produit **tous les artefacts du papier** :

| Bloc | Contenu | Sortie |
|---|---|---|
| A | Statistiques 4 intervalles, chronologie des attaques | Table 1, Figure 1 |
| B | Splits gelés (5 stratifiés + chronologique), sondes temporelles | chiffre d'accroche RQ2 |
| C | 9 modèles × conditions `full`/`clean` (avant audit) | base E2 |
| D | Audit : importance par permutation + test de transférabilité mono-feature → **liste noire** | artefact clé RQ2 |
| E | Condition `audited` (après liste noire) + split chrono | figure avant/après |
| F | Autoencodeur (bénin seul) : AUROC global + par famille | section « attaque inconnue » |
| G | Calibration : température, ECE, diagrammes de fiabilité | E5 |
| H | Multi-intervalles 5/10/30 s (LightGBM, XGBoost, DNN) | E4 |
| I | Banc de coût CPU : latence batch-1, débit batch-512, tailles | E6 |
| J | Statistique : McNemar + Holm, bootstrap | rigueur Q1 |
| K | **Figures 300 dpi + tables LaTeX + JSON complet** | à me renvoyer |

**Exécution.** Runtime **GPU**. Durée totale ≈ 6–8 h : le notebook **sauvegarde après chaque run** dans
`MyDrive/GeNIS/article1_master/` et **reprend automatiquement** où il s'était arrêté —
vous pouvez l'exécuter en 2–3 sessions (Exécution → Tout exécuter à chaque fois).

**À me renvoyer à la fin :** `article1_results.json` + `article1_figures_tables.zip` (téléchargés automatiquement).


In [ ]:
# 1) Environnement
import os, sys, glob, json, time, math, shutil, hashlib, pathlib, platform, itertools
import numpy as np, pandas as pd, sklearn, scipy
from scipy import stats as sps
import tensorflow as tf
import matplotlib
import matplotlib.pyplot as plt
try:
    import xgboost, lightgbm
except ImportError:
    %pip -q install xgboost lightgbm
    import xgboost, lightgbm

plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "font.size": 9})
print("python", platform.python_version(), "| tf", tf.__version__,
      "| sklearn", sklearn.__version__, "| xgb", xgboost.__version__,
      "| lgbm", lightgbm.__version__)
GPU = bool(tf.config.list_physical_devices("GPU"))
print("GPU :", GPU or "AUCUN (les blocs profonds seront lents)")


In [ ]:
# 2) Drive, donnees, reprise
from google.colab import drive
drive.mount('/content/drive')

WORK = pathlib.Path("/content/genis"); WORK.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)
SAVE = pathlib.Path("/content/drive/MyDrive/GeNIS/article1_master")
PROBS = SAVE / "probs"; FIGS = SAVE / "figures"; TABS = SAVE / "tables"
for p in (SAVE, PROBS, FIGS, TABS): p.mkdir(parents=True, exist_ok=True)

RES_PATH = SAVE / "article1_results.json"
RESULTS = json.loads(RES_PATH.read_text()) if RES_PATH.exists() else {}
RESULTS.setdefault("models", {}); RESULTS.setdefault("meta", {})
RESULTS["meta"].update({"updated": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
                        "env": {"tf": tf.__version__, "sklearn": sklearn.__version__,
                                "xgb": xgboost.__version__, "lgbm": lightgbm.__version__,
                                "gpu": GPU}})
def save_results():
    RES_PATH.write_text(json.dumps(RESULTS, indent=1, default=float), encoding="utf-8")
print("runs deja calcules :", len(RESULTS["models"]))

drive_zip = "/content/drive/MyDrive/GeNIS/2-flows.zip"
if not pathlib.Path("flows").exists():
    if pathlib.Path(drive_zip).exists():
        print("copie depuis Drive…"); shutil.copy(drive_zip, "2-flows.zip")
    else:
        print("telechargement Zenodo (~380 Mo)…")
        !wget -q --show-progress "https://zenodo.org/records/14919237/files/2-flows.zip?download=1" -O 2-flows.zip
    !unzip -o -q 2-flows.zip -d flows
csvs = sorted(glob.glob("flows/**/*.csv", recursive=True))
assert csvs, "aucun CSV"
print(len(csvs), "CSV")


In [ ]:
# 3) Bloc A1 — statistiques des 4 intervalles (Table 1 du papier)
INTERVALS = ["5", "10", "30", "60"]
if "interval_stats" not in RESULTS:
    stats_iv = {}
    for iv in INTERVALS:
        counts = {pathlib.Path(f).stem: sum(1 for _ in open(f, "rb")) - 1
                  for f in csvs if f"flows-{iv}-sec" in f}
        total = sum(counts.values())
        benign = sum(v for k, v in counts.items() if k.startswith("benign"))
        stats_iv[iv] = {"files": counts, "total": total, "benign": benign,
                        "benign_share": benign / total}
    RESULTS["interval_stats"] = stats_iv; save_results()
for iv in INTERVALS:
    s = RESULTS["interval_stats"][iv]
    print(f"{iv:>2s} s : {s['total']:>9,} flux | benin {s['benign']:>7,} ({s['benign_share']:.1%})")


In [ ]:
# 4) Bloc A2 — tranche 60 s : labels 9 classes, features explicites, chronologie
def load_slice(iv):
    files = [c for c in csvs if f"flows-{iv}-sec" in c]
    d = pd.concat([pd.read_csv(c, low_memory=False) for c in files], ignore_index=True)
    ysub = d["SubCategoryLabel"].astype(str).str.strip()
    y9 = ysub.where(~ysub.str.startswith("benign"), "benign")
    t = pd.to_numeric(d["StartTime"], errors="coerce")
    assert t.notna().all()
    return d, y9, t

IDENT_LIST = ["FlowID", "AutoId", "SrcAddr", "DstAddr", "Ssaddr", "Sdaddr",
              "SrcMac", "DstMac", "SrcOui", "DstOui", "Sport", "Dport",
              "sIpId", "dIpId", "sMpls", "dMpls", "sAS", "dAS", "iAS",
              "sCo", "dCo", "sVid", "dVid"]
POS_LIST   = ["StartTime", "LastTime", "Rank", "Seq"]
LAB_LIST   = ["BinaryLabel", "CategoryLabel", "SubCategoryLabel"]

def feature_sets(d):
    ident = [c for c in IDENT_LIST if c in d.columns]
    pos   = [c for c in POS_LIST if c in d.columns]
    num = (d.drop(columns=ident + LAB_LIST, errors="ignore")
             .select_dtypes(include=[np.number])
             .replace([np.inf, -np.inf], np.nan))
    const = num.nunique(dropna=True); num = num.drop(columns=const[const <= 1].index)
    num = num.fillna(0.0).astype(np.float32)
    full = list(num.columns); clean = [c for c in full if c not in pos]
    return num, full, clean, pos, ident

df, y9_raw, t_start = load_slice("60")
X_all, F_FULL, F_CLEAN, POSITIONAL, IDENTIFIERS = feature_sets(df)
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder(); y = le.fit_transform(y9_raw)
CLASS_NAMES = list(le.classes_); C = len(CLASS_NAMES)
BENIGN_IDX = CLASS_NAMES.index("benign")
print("flux :", len(y), "| classes :", CLASS_NAMES)
print(f"features full {len(F_FULL)} | clean {len(F_CLEAN)} | positionnelles {POSITIONAL}")
RESULTS["slice60"] = {"n": int(len(y)), "classes": CLASS_NAMES,
                      "class_counts": pd.Series(y9_raw).value_counts().to_dict(),
                      "benign_share": float((y == BENIGN_IDX).mean()),
                      "features_full": F_FULL, "features_clean": F_CLEAN,
                      "positional": POSITIONAL, "identifiers_excluded": IDENTIFIERS}

# Figure 1 : chronologie des fenetres d'attaque (echantillonnee)
t0 = t_start.min()
fig, ax = plt.subplots(figsize=(9, 3.2))
rng = np.random.RandomState(0)
for i, cn in enumerate(CLASS_NAMES):
    ix = np.where(y == i)[0]
    ix = rng.choice(ix, size=min(3000, len(ix)), replace=False)
    ax.plot((t_start.values[ix] - t0) / 3600, np.full(len(ix), i), "|", ms=4,
            color=("tab:green" if i == BENIGN_IDX else "tab:red"), alpha=0.25)
ax.set_yticks(range(C)); ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel("heures depuis le debut de capture"); ax.set_title("Fenetres temporelles par classe (60 s)")
plt.tight_layout(); plt.savefig(FIGS / "fig1_timeline.png"); plt.savefig(FIGS / "fig1_timeline.pdf"); plt.show()
save_results()


In [ ]:
# 5) Bloc B — splits geles + table chrono + sondes temporelles
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

SEEDS = [1, 2, 3, 4, 5]
SPLITS_PATH = SAVE / "frozen_splits_60s.npz"
if SPLITS_PATH.exists():
    z = np.load(SPLITS_PATH)
    splits = {k: (z[f"{k}_train"], z[f"{k}_val"], z[f"{k}_test"])
              for k in [f"strat_seed{s}" for s in SEEDS] + ["chrono"]}
    print("splits recharges (geles)")
else:
    idx = np.arange(len(y)); splits = {}
    for s in SEEDS:
        itr, itmp = train_test_split(idx, test_size=0.40, random_state=s, stratify=y)
        iva, ite = train_test_split(itmp, test_size=0.50, random_state=s, stratify=y[itmp])
        splits[f"strat_seed{s}"] = (itr, iva, ite)
    o = np.argsort(t_start.values, kind="stable"); n = len(o)
    splits["chrono"] = (o[:int(.6*n)], o[int(.6*n):int(.8*n)], o[int(.8*n):])
    np.savez_compressed(SPLITS_PATH, **{f"{k}_{p}": v for k, (a, b, c) in splits.items()
                                        for p, v in zip(("train", "val", "test"), (a, b, c))})
    print("splits generes et geles ->", SPLITS_PATH)

tab = pd.DataFrame({p: pd.Series(y[ix]).value_counts().reindex(range(C), fill_value=0).values
                    for p, ix in zip(("train", "val", "test"), splits["chrono"])}, index=CLASS_NAMES)
print("\nClasses dans le split chronologique :\n", tab)
RESULTS["chrono_class_table"] = tab.to_dict()

def probe(cols, key):
    tr, va, te = splits[key]
    Xp = df[cols].astype(np.float64).values
    clf = DecisionTreeClassifier(random_state=1).fit(Xp[tr], y[tr])
    return float(accuracy_score(y[te], clf.predict(Xp[te])))

if "shortcut_probes" not in RESULTS:
    chance = float(pd.Series(y).value_counts(normalize=True).max())
    pr = {"chance_majority": chance}
    for nm, cols in [("starttime_only", ["StartTime"]), ("positional_all", POSITIONAL)]:
        accs = [probe(cols, f"strat_seed{s}") for s in SEEDS]
        pr[nm] = {"strat_mean": float(np.mean(accs)), "strat_std": float(np.std(accs)),
                  "chrono": probe(cols, "chrono")}
    RESULTS["shortcut_probes"] = pr; save_results()
for k, v in RESULTS["shortcut_probes"].items():
    print(k, ":", v)


In [ ]:
# 6) Helpers : scaling, metriques, alignement de classes, entrainement
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import (f1_score, matthews_corrcoef, average_precision_score,
                             roc_auc_score, confusion_matrix)

def make_xy(cols, key):
    tr, va, te = splits[key]
    X = X_all[cols].values
    sc = RobustScaler().fit(X[tr])
    parts = [np.nan_to_num(sc.transform(X[ix]), nan=0., posinf=0., neginf=0.).astype(np.float32)
             for ix in (tr, va, te)]
    return parts, (y[tr], y[va], y[te])

def align_probs(classes_, probs):
    if len(classes_) == C and list(classes_) == list(range(C)):
        return probs
    out = np.zeros((len(probs), C), dtype=np.float32)
    out[:, np.asarray(classes_, int)] = probs
    return out

def evaluate(y_true, probs, fit_t, pred_t):
    pred = probs.argmax(1)
    present = np.unique(y_true)
    is_att = y_true != BENIGN_IDX; pred_att = pred != BENIGN_IDX
    p_att = 1.0 - probs[:, BENIGN_IDX]
    return {"accuracy": float((pred == y_true).mean()),
            "macro_f1": float(f1_score(y_true, pred, labels=present, average="macro", zero_division=0)),
            "mcc": float(matthews_corrcoef(y_true, pred)),
            "per_class_f1": {CLASS_NAMES[i]: float(v) for i, v in zip(range(C),
                f1_score(y_true, pred, labels=range(C), average=None, zero_division=0))},
            "binary": {"detection_f1": float(f1_score(is_att, pred_att, zero_division=0)),
                       "fpr": float(pred_att[~is_att].mean()) if (~is_att).any() else None,
                       "fnr": float((~pred_att[is_att]).mean()) if is_att.any() else None,
                       "pr_auc": float(average_precision_score(is_att, p_att))
                                 if 0 < is_att.mean() < 1 else None},
            "fit_time_s": round(fit_t, 2), "predict_time_s": round(pred_t, 3)}

def done(key): return key in RESULTS["models"] and (PROBS / f"{key.replace('|','_')}.npz").exists()

def record(key, y_true, pva, pte, fit_t, pred_t, ypred_te=None):
    RESULTS["models"][key] = evaluate(y_true, pte, fit_t, pred_t)
    np.savez_compressed(PROBS / f"{key.replace('|','_')}.npz",
                        probs_val=pva.astype(np.float16), probs_test=pte.astype(np.float16))
    r = RESULTS["models"][key]
    fpr = r["binary"]["fpr"]; fs = f"{fpr:.4%}" if fpr is not None else "n/a"
    print(f"{key:42s} acc {r['accuracy']:.4f} mF1 {r['macro_f1']:.4f} "
          f"MCC {r['mcc']:.4f} FPR {fs} [{fit_t:.0f}s]")
    save_results()
print("helpers OK")


In [ ]:
# 7) Constructeurs de modeles (sklearn + Keras, dont FT-Transformer)
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from tensorflow.keras import layers, models, callbacks
from sklearn.utils.class_weight import compute_class_weight

def sk_zoo():
    return {"majority": DummyClassifier(strategy="most_frequent"),
            "logreg":   LogisticRegression(max_iter=1000, n_jobs=-1),
            "rf":       RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=0),
            "xgboost":  XGBClassifier(tree_method="hist", n_estimators=300, max_depth=8,
                                      learning_rate=0.1, n_jobs=-1, random_state=0,
                                      eval_metric="mlogloss"),
            "lightgbm": LGBMClassifier(n_estimators=300, num_leaves=63, learning_rate=0.1,
                                       n_jobs=-1, random_state=0, verbose=-1)}
FAST = list(sk_zoo().keys())

# --- trio BAg-IDS : architectures STRICTEMENT identiques au papier systeme ---
def build_dnn(F):
    return models.Sequential([layers.Input((F,)),
        layers.Dense(128, activation="relu"), layers.Dropout(0.3),
        layers.Dense(64, activation="relu"), layers.Dropout(0.2),
        layers.Dense(C, activation="softmax")], name="dnn")
def build_cnn(F):
    return models.Sequential([layers.Input((F, 1)),
        layers.Conv1D(64, 3, activation="relu", padding="same"), layers.MaxPooling1D(2),
        layers.Conv1D(32, 3, activation="relu", padding="same"), layers.Flatten(),
        layers.Dense(64, activation="relu"), layers.Dropout(0.3),
        layers.Dense(C, activation="softmax")], name="cnn")
def build_rnn(F):
    return models.Sequential([layers.Input((F, 1)),
        layers.SimpleRNN(64, activation="relu"), layers.Dropout(0.3),
        layers.Dense(64, activation="relu"),
        layers.Dense(C, activation="softmax")], name="rnn")

# --- FT-Transformer compact (Gorishniy et al. 2021), auto-contenu ---
class FeatureTokenizer(layers.Layer):
    def __init__(self, d, **kw): super().__init__(**kw); self.d = d
    def build(self, shape):
        F = int(shape[-1])
        self.w = self.add_weight(shape=(F, self.d), initializer="glorot_uniform", name="w")
        self.b = self.add_weight(shape=(F, self.d), initializer="zeros", name="b")
    def call(self, x): return x[:, :, None] * self.w + self.b

class ClsToken(layers.Layer):
    def __init__(self, d, **kw): super().__init__(**kw); self.d = d
    def build(self, shape):
        self.cls = self.add_weight(shape=(1, 1, self.d), initializer="glorot_uniform", name="cls")
    def call(self, x):
        b = tf.shape(x)[0]
        return tf.concat([tf.tile(self.cls, [b, 1, 1]), x], axis=1)

def build_ftt(F, d=64, heads=8, blocks=3, ff=128, drop=0.1):
    inp = layers.Input((F,))
    x = FeatureTokenizer(d)(inp)
    x = ClsToken(d)(x)
    for _ in range(blocks):
        h = layers.LayerNormalization()(x)
        h = layers.MultiHeadAttention(num_heads=heads, key_dim=d // heads, dropout=drop)(h, h)
        x = layers.Add()([x, h])
        h = layers.LayerNormalization()(x)
        h = layers.Dense(ff, activation="gelu")(h); h = layers.Dropout(drop)(h)
        h = layers.Dense(d)(h)
        x = layers.Add()([x, h])
    cls = layers.LayerNormalization()(x[:, 0])
    out = layers.Dense(C, activation="softmax")(cls)
    return models.Model(inp, out, name="ftt")

DEEP = {"rnn": build_rnn, "cnn": build_cnn, "dnn": build_dnn, "ftt": build_ftt}
def shape_for(name, A): return A if name in ("dnn", "ftt") else A.reshape(-1, A.shape[1], 1)

def class_weights_safe(ytr):
    present = np.unique(ytr)
    w = compute_class_weight("balanced", classes=present, y=ytr)
    cw = {int(c): 1.0 for c in range(C)}; cw.update({int(c): float(v) for c, v in zip(present, w)})
    return cw

def train_deep(mname, cond_cols, key_split, seed, tag):
    key = f"{mname}|{tag}|{key_split}"
    if done(key): return
    (Xtr, Xva, Xte), (ytr, yva, yte) = make_xy(cond_cols, key_split)
    tf.keras.utils.set_random_seed(seed)
    m = DEEP[mname](Xtr.shape[1])
    lr = 5e-4 if mname == "ftt" else 1e-3
    m.compile(optimizer=tf.keras.optimizers.Adam(lr),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    t0 = time.time()
    m.fit(shape_for(mname, Xtr), ytr, validation_data=(shape_for(mname, Xva), yva),
          epochs=30, batch_size=512 if mname == "ftt" else 256,
          class_weight=class_weights_safe(ytr), verbose=0,
          callbacks=[callbacks.EarlyStopping(monitor="val_loss", patience=5,
                                             restore_best_weights=True)])
    fit_t = time.time() - t0
    t0 = time.time(); pte = m.predict(shape_for(mname, Xte), batch_size=1024, verbose=0)
    pred_t = time.time() - t0
    pva = m.predict(shape_for(mname, Xva), batch_size=1024, verbose=0)
    if tag == "clean" and key_split == "strat_seed1":
        m.save(SAVE / f"{mname}_clean_seed1.keras")
    record(key, yte, pva, pte, fit_t, pred_t)

def train_fast(mname, cond_cols, key_split, tag):
    key = f"{mname}|{tag}|{key_split}"
    if done(key): return
    (Xtr, Xva, Xte), (ytr, yva, yte) = make_xy(cond_cols, key_split)
    model = sk_zoo()[mname]
    t0 = time.time(); model.fit(Xtr, ytr); fit_t = time.time() - t0
    t0 = time.time(); pte = align_probs(model.classes_, model.predict_proba(Xte)); pred_t = time.time() - t0
    pva = align_probs(model.classes_, model.predict_proba(Xva))
    record(key, yte, pva, pte, fit_t, pred_t)
print("constructeurs OK")


In [ ]:
# 8) Bloc C — grille avant-audit : full + clean
STRATS = [f"strat_seed{s}" for s in SEEDS]
COND = {"full": F_FULL, "clean": F_CLEAN}

for tag, cols in COND.items():
    for sk_ in STRATS + ["chrono"]:
        for mname in FAST:
            train_fast(mname, cols, sk_, tag)

# profond : clean x 5 graines + clean chrono ; full graine 1 (illustration avant-audit)
for mname in DEEP:
    for s in SEEDS:
        train_deep(mname, F_CLEAN, f"strat_seed{s}", s, "clean")
    train_deep(mname, F_CLEAN, "chrono", 1, "clean")
    train_deep(mname, F_FULL, "strat_seed1", 1, "full")
print("bloc C termine")


In [ ]:
# 9) Bloc D — audit : importance par permutation + transferabilite mono-feature
from sklearn.inspection import permutation_importance
from sklearn.tree import DecisionTreeClassifier

if "audit" not in RESULTS:
    (Xtr, Xva, Xte), (ytr, yva, yte) = make_xy(F_CLEAN, "strat_seed1")
    ref = LGBMClassifier(n_estimators=300, num_leaves=63, learning_rate=0.1,
                         n_jobs=-1, random_state=0, verbose=-1).fit(Xtr, ytr)
    rng = np.random.RandomState(0)
    sub = rng.choice(len(yte), size=min(20000, len(yte)), replace=False)
    imp = permutation_importance(ref, Xte[sub], yte[sub], n_repeats=5,
                                 random_state=0, n_jobs=-1, scoring="accuracy")
    order = np.argsort(-imp.importances_mean)
    top = [(F_CLEAN[i], float(imp.importances_mean[i])) for i in order[:15]]

    # Test de transferabilite : une feature dont le pouvoir predictif ne survit pas
    # au split chronologique encode la POSITION, pas le comportement -> liste noire.
    chance = float(pd.Series(y).value_counts(normalize=True).max())
    flagged, table = [], []
    for feat, im in top:
        a_strat = probe([feat], "strat_seed1")
        a_chrono = probe([feat], "chrono")
        shortcut = (a_strat > 3 * chance) and (a_chrono < 0.5 * a_strat)
        table.append({"feature": feat, "perm_importance": im,
                      "single_acc_strat": a_strat, "single_acc_chrono": a_chrono,
                      "flagged": bool(shortcut)})
        if shortcut: flagged.append(feat)
        print(f"{feat:22s} imp {im:.4f} | seule: strat {a_strat:.3f} chrono {a_chrono:.3f}"
              + ("  <-- RACCOURCI" if shortcut else ""))
    BLACKLIST = POSITIONAL + flagged
    RESULTS["audit"] = {"perm_importance_top": top, "transfer_table": table,
                        "blacklist": BLACKLIST}
    save_results()
BLACKLIST = RESULTS["audit"]["blacklist"]
F_AUDIT = [c for c in F_CLEAN if c not in BLACKLIST]
RESULTS["slice60"]["features_audited"] = F_AUDIT
print(f"\nLISTE NOIRE ({len(BLACKLIST)}) : {BLACKLIST}")
print(f"features auditees : {len(F_AUDIT)}")
save_results()


In [ ]:
# 10) Bloc E — condition auditee (apres liste noire), stratifie + chrono
for sk_ in STRATS + ["chrono"]:
    for mname in FAST:
        train_fast(mname, F_AUDIT, sk_, "audited")
for mname in DEEP:
    for s in SEEDS:
        train_deep(mname, F_AUDIT, f"strat_seed{s}", s, "audited")
    train_deep(mname, F_AUDIT, "chrono", 1, "audited")
print("bloc E termine")


In [ ]:
# 11) Bloc F — autoencodeur (benin seul) : AUROC global + par famille
from sklearn.metrics import roc_auc_score

def build_ae(F):
    return models.Sequential([layers.Input((F,)),
        layers.Dense(64, activation="relu"), layers.Dense(32, activation="relu"),
        layers.Dense(16, activation="relu"), layers.Dense(32, activation="relu"),
        layers.Dense(64, activation="relu"), layers.Dense(F)], name="ae")

if "autoencoder" not in RESULTS:
    ae_res = {}
    for s in SEEDS:
        (Xtr, Xva, Xte), (ytr, yva, yte) = make_xy(F_AUDIT, f"strat_seed{s}")
        tf.keras.utils.set_random_seed(s)
        ben_tr = Xtr[ytr == BENIGN_IDX]; ben_va = Xva[yva == BENIGN_IDX]
        ae = build_ae(Xtr.shape[1])
        ae.compile(optimizer="adam", loss="mse")
        ae.fit(ben_tr, ben_tr, validation_data=(ben_va, ben_va), epochs=50,
               batch_size=256, verbose=0,
               callbacks=[callbacks.EarlyStopping(monitor="val_loss", patience=5,
                                                  restore_best_weights=True)])
        err = np.mean((ae.predict(Xte, batch_size=1024, verbose=0) - Xte) ** 2, axis=1)
        is_att = yte != BENIGN_IDX
        per_fam = {}
        for i, cn in enumerate(CLASS_NAMES):
            if i == BENIGN_IDX: continue
            m_ = (yte == i) | (yte == BENIGN_IDX)
            if (yte[m_] == i).any():
                per_fam[cn] = float(roc_auc_score(yte[m_] == i, err[m_]))
        ae_res[f"seed{s}"] = {"auroc_global": float(roc_auc_score(is_att, err)),
                              "auroc_per_family": per_fam}
        print(f"AE seed {s} : AUROC {ae_res[f'seed{s}']['auroc_global']:.4f}")
    RESULTS["autoencoder"] = ae_res; save_results()
g = [v["auroc_global"] for v in RESULTS["autoencoder"].values()]
print(f"AE AUROC global : {np.mean(g):.4f} +/- {np.std(g):.4f}")


In [ ]:
# 12) Bloc G — calibration : temperature scaling + ECE + diagrammes de fiabilite
# NB : softmax(log(p)/T) == softmax(z/T) (invariance au decalage par ligne),
# donc la calibration se calcule exactement depuis les probabilites sauvegardees.
from scipy.optimize import minimize_scalar

def load_probs(key):
    z = np.load(PROBS / f"{key.replace('|','_')}.npz")
    return z["probs_val"].astype(np.float64), z["probs_test"].astype(np.float64)

def fit_temperature(pva, yva):
    logp = np.log(np.clip(pva, 1e-12, 1))
    def nll(T):
        q = logp / T; q -= q.max(1, keepdims=True)
        p = np.exp(q); p /= p.sum(1, keepdims=True)
        return -np.mean(np.log(np.clip(p[np.arange(len(yva)), yva], 1e-12, 1)))
    return float(minimize_scalar(nll, bounds=(0.05, 10.0), method="bounded").x)

def apply_T(p, T):
    q = np.log(np.clip(p, 1e-12, 1)) / T; q -= q.max(1, keepdims=True)
    e = np.exp(q); return e / e.sum(1, keepdims=True)

def ece(p, y_true, bins=15):
    conf = p.max(1); pred = p.argmax(1); acc = (pred == y_true).astype(float)
    edges = np.linspace(0, 1, bins + 1); out = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any(): out += m.mean() * abs(acc[m].mean() - conf[m].mean())
    return float(out)

MAIN = ["rf", "xgboost", "lightgbm", "rnn", "cnn", "dnn", "ftt", "logreg"]
if "calibration" not in RESULTS:
    cal = {}
    _, _, yte_ref = (y[splits["strat_seed1"][0]], y[splits["strat_seed1"][1]], y[splits["strat_seed1"][2]])
    yva_ref = y[splits["strat_seed1"][1]]
    for mname in MAIN:
        key = f"{mname}|audited|strat_seed1"
        if not done(key): continue
        pva, pte = load_probs(key)
        T = fit_temperature(pva, yva_ref)
        cal[mname] = {"T": T, "ece_before": ece(pte, yte_ref),
                      "ece_after": ece(apply_T(pte, T), yte_ref)}
        print(f"{mname:9s} T={T:.2f}  ECE {cal[mname]['ece_before']:.4f} -> {cal[mname]['ece_after']:.4f}")
    RESULTS["calibration"] = cal; save_results()

# diagrammes de fiabilite (2 x 4)
yte_ref = y[splits["strat_seed1"][2]]
fig, axes = plt.subplots(2, 4, figsize=(11, 5), sharex=True, sharey=True)
for ax, mname in zip(axes.flat, MAIN):
    key = f"{mname}|audited|strat_seed1"
    if not done(key) or mname not in RESULTS["calibration"]:
        ax.axis("off"); continue
    pva, pte = load_probs(key); T = RESULTS["calibration"][mname]["T"]
    for p, lab in [(pte, "brut"), (apply_T(pte, T), "calibre")]:
        conf, pred = p.max(1), p.argmax(1)
        edges = np.linspace(0, 1, 16); xs, ys = [], []
        for lo, hi in zip(edges[:-1], edges[1:]):
            m = (conf > lo) & (conf <= hi)
            if m.any(): xs.append(conf[m].mean()); ys.append((pred[m] == yte_ref[m]).mean())
        ax.plot(xs, ys, marker="o", ms=2.5, label=lab)
    ax.plot([0, 1], [0, 1], "k--", lw=.6)
    ax.set_title(mname, fontsize=8)
axes[0, 0].legend(fontsize=7)
fig.suptitle("Diagrammes de fiabilite (condition auditee, graine 1)")
plt.tight_layout(); plt.savefig(FIGS / "fig4_reliability.png"); plt.savefig(FIGS / "fig4_reliability.pdf"); plt.show()


In [ ]:
# 13) Bloc H — multi-intervalles : 5 / 10 / 30 s (LightGBM, XGBoost, DNN)
IV_MODELS_FAST = ["lightgbm", "xgboost"]; IV_SEEDS = [1, 2, 3]
RESULTS.setdefault("intervals", {})
for iv in ["5", "10", "30"]:
    if iv in RESULTS["intervals"]: continue
    print(f"===== intervalle {iv} s =====")
    d_iv, y9_iv, t_iv = load_slice(iv)
    Xiv, fullc, cleanc, pos_iv, _ = feature_sets(d_iv)
    audc = [c for c in cleanc if c not in BLACKLIST]
    y_iv = le.transform(y9_iv)
    res_iv = {"n": int(len(y_iv)),
              "benign_share": float((y_iv == BENIGN_IDX).mean()), "runs": {}}
    # sonde temporelle par intervalle
    from sklearn.tree import DecisionTreeClassifier
    idx = np.arange(len(y_iv))
    itr, itmp = train_test_split(idx, test_size=0.4, random_state=1, stratify=y_iv)
    iva_, ite_ = train_test_split(itmp, test_size=0.5, random_state=1, stratify=y_iv[itmp])
    stp = d_iv["StartTime"].astype(np.float64).values.reshape(-1, 1)
    dtc = DecisionTreeClassifier(random_state=1).fit(stp[itr], y_iv[itr])
    res_iv["probe_starttime"] = float((dtc.predict(stp[ite_]) == y_iv[ite_]).mean())
    for s in IV_SEEDS:
        itr, itmp = train_test_split(idx, test_size=0.4, random_state=s, stratify=y_iv)
        iva_, ite_ = train_test_split(itmp, test_size=0.5, random_state=s, stratify=y_iv[itmp])
        Xa = Xiv[audc].values
        sc = RobustScaler().fit(Xa[itr])
        Xtr, Xte = [np.nan_to_num(sc.transform(Xa[i_]), nan=0., posinf=0., neginf=0.).astype(np.float32)
                    for i_ in (itr, ite_)]
        ytr_, yte_ = y_iv[itr], y_iv[ite_]
        for mname in IV_MODELS_FAST:
            mdl = sk_zoo()[mname]
            t0 = time.time(); mdl.fit(Xtr, ytr_); ft = time.time() - t0
            t0 = time.time(); pte = align_probs(mdl.classes_, mdl.predict_proba(Xte)); pt = time.time() - t0
            res_iv["runs"][f"{mname}|seed{s}"] = evaluate(yte_, pte, ft, pt)
        # DNN (architecture BAg-IDS)
        tf.keras.utils.set_random_seed(s)
        m = build_dnn(Xtr.shape[1])
        m.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
        Xva_ = np.nan_to_num(sc.transform(Xa[iva_]), nan=0., posinf=0., neginf=0.).astype(np.float32)
        t0 = time.time()
        m.fit(Xtr, ytr_, validation_data=(Xva_, y_iv[iva_]), epochs=30, batch_size=256,
              class_weight=class_weights_safe(ytr_), verbose=0,
              callbacks=[callbacks.EarlyStopping(monitor="val_loss", patience=5,
                                                 restore_best_weights=True)])
        ft = time.time() - t0
        t0 = time.time(); pte = m.predict(Xte, batch_size=1024, verbose=0); pt = time.time() - t0
        res_iv["runs"][f"dnn|seed{s}"] = evaluate(yte_, pte, ft, pt)
        print(f"  seed {s} ok")
    RESULTS["intervals"][iv] = res_iv; save_results()
    del d_iv, Xiv
print("bloc H termine")


In [ ]:
# 14) Bloc I — banc de cout CPU (condition auditee, graine 1)
if "cost" not in RESULTS:
    cost = {}
    (Xtr, Xva, Xte), (ytr, yva, yte) = make_xy(F_AUDIT, "strat_seed1")
    n1, nb = 200, 20   # 200 lat. batch-1 ; 20 batchs de 512
    x1 = Xte[:n1]; xb = Xte[:512]
    for mname in FAST:
        mdl = sk_zoo()[mname]; mdl.fit(Xtr, ytr)
        lat = []
        for i in range(n1):
            t0 = time.perf_counter(); mdl.predict_proba(x1[i:i+1]); lat.append(time.perf_counter() - t0)
        t0 = time.perf_counter()
        for _ in range(nb): mdl.predict_proba(xb)
        thr = nb * 512 / (time.perf_counter() - t0)
        import pickle, io
        buf = io.BytesIO(); pickle.dump(mdl, buf)
        cost[mname] = {"lat_p50_ms": float(np.percentile(lat, 50) * 1e3),
                       "lat_p99_ms": float(np.percentile(lat, 99) * 1e3),
                       "throughput_512": float(thr), "size_mb": buf.getbuffer().nbytes / 1e6}
        print(mname, cost[mname])
    (Xtr_c, _, Xte_c), _ = make_xy(F_CLEAN, "strat_seed1")   # les .keras sont entraines sur CLEAN
    x1c, xbc = Xte_c[:n1], Xte_c[:512]
    with tf.device("/CPU:0"):
        for mname in DEEP:
            path = SAVE / f"{mname}_clean_seed1.keras"
            if not path.exists(): continue
            m = tf.keras.models.load_model(path, compile=False)
            x1, xb = x1c, xbc
            xin = lambda a: a if mname in ("dnn", "ftt") else a.reshape(-1, a.shape[1], 1)
            m.predict(xin(x1[:8]), verbose=0)          # warm-up
            lat = []
            for i in range(n1):
                t0 = time.perf_counter(); m.predict(xin(x1[i:i+1]), verbose=0)
                lat.append(time.perf_counter() - t0)
            t0 = time.perf_counter()
            for _ in range(nb): m.predict(xin(xb), verbose=0)
            thr = nb * 512 / (time.perf_counter() - t0)
            cost[mname] = {"lat_p50_ms": float(np.percentile(lat, 50) * 1e3),
                           "lat_p99_ms": float(np.percentile(lat, 99) * 1e3),
                           "throughput_512": float(thr),
                           "size_mb": path.stat().st_size / 1e6,
                           "params": int(m.count_params())}
            print(mname, cost[mname])
    RESULTS["cost"] = cost; save_results()
print("bloc I termine")


In [ ]:
# 15) Bloc J — statistique : McNemar apparie + Holm, bootstrap 95%
from scipy.stats import chi2 as chi2dist

def mcnemar_p(pred_a, pred_b, y_true):
    ca, cb = pred_a == y_true, pred_b == y_true
    b_ = int((ca & ~cb).sum()); c_ = int((~ca & cb).sum())
    if b_ + c_ == 0: return 1.0
    stat = (abs(b_ - c_) - 1) ** 2 / (b_ + c_)
    return float(chi2dist.sf(stat, 1))

if "stats" not in RESULTS:
    yte_ref = y[splits["strat_seed1"][2]]
    preds = {}
    for mname in MAIN:
        key = f"{mname}|audited|strat_seed1"
        if done(key): preds[mname] = load_probs(key)[1].argmax(1)
    pairs = list(itertools.combinations(sorted(preds), 2))
    raw = {f"{a}|{b}": mcnemar_p(preds[a], preds[b], yte_ref) for a, b in pairs}
    order = sorted(raw, key=raw.get)          # correction de Holm
    m_ = len(order); holm = {}
    for i, k in enumerate(order):
        holm[k] = min(1.0, raw[k] * (m_ - i))
    boot = {}
    rng = np.random.RandomState(0); B = 1000
    for mname, pr in preds.items():
        vals = []
        for _ in range(B):
            ix = rng.randint(0, len(yte_ref), len(yte_ref))
            vals.append(f1_score(yte_ref[ix], pr[ix], average="macro", zero_division=0))
        boot[mname] = {"macro_f1_ci95": [float(np.percentile(vals, 2.5)),
                                         float(np.percentile(vals, 97.5))]}
    RESULTS["stats"] = {"mcnemar_raw": raw, "mcnemar_holm": holm, "bootstrap": boot}
    save_results()
sig = {k: v for k, v in RESULTS["stats"]["mcnemar_holm"].items() if v < 0.05}
print(f"paires significatives (Holm<0.05) : {len(sig)}/{len(RESULTS['stats']['mcnemar_holm'])}")
for k, v in list(RESULTS["stats"]["bootstrap"].items()):
    print(f"{k:9s} macro-F1 CI95 {v['macro_f1_ci95']}")


In [ ]:
# 16) Bloc K — figures finales + tables LaTeX + archive
ALL_MODELS = FAST + list(DEEP)

def agg(mname, tag):
    keys = [f"{mname}|{tag}|strat_seed{s}" for s in SEEDS]
    vals = [RESULTS["models"][k]["macro_f1"] for k in keys if k in RESULTS["models"]]
    return (np.mean(vals), np.std(vals)) if vals else (np.nan, 0)

def one(mname, tag, split):
    k = f"{mname}|{tag}|{split}"
    return RESULTS["models"][k]["macro_f1"] if k in RESULTS["models"] else np.nan

# --- Figure 2 : sonde + avant/apres (slopegraph macro-F1) ---
conds = ["full", "clean", "audited", "audited-chrono"]
fig, ax = plt.subplots(figsize=(7, 4.2))
for mname in ALL_MODELS:
    if mname == "majority": continue
    ys = [agg(mname, "full")[0] if mname in FAST else one(mname, "full", "strat_seed1"),
          agg(mname, "clean")[0], agg(mname, "audited")[0],
          one(mname, "audited", "chrono")]
    ax.plot(range(4), ys, marker="o", ms=3, label=mname)
ax.set_xticks(range(4)); ax.set_xticklabels(["full\n(avant)", "clean", "audited\n(liste noire)", "audited\n+ chrono"])
ax.set_ylabel("macro-F1"); ax.set_title("Classement avant / apres audit — 60 s")
ax.legend(fontsize=7, ncol=2)
plt.tight_layout(); plt.savefig(FIGS / "fig2_before_after.png"); plt.savefig(FIGS / "fig2_before_after.pdf"); plt.show()

# --- Figure 3 : detectabilite par famille vs intervalle (LightGBM audite) ---
det = pd.DataFrame(index=CLASS_NAMES, columns=INTERVALS, dtype=float)
for iv in INTERVALS:
    if iv == "60":
        pcf = RESULTS["models"].get("lightgbm|audited|strat_seed1", {}).get("per_class_f1", {})
    else:
        runs = RESULTS.get("intervals", {}).get(iv, {}).get("runs", {})
        pcs = [r["per_class_f1"] for k, r in runs.items() if k.startswith("lightgbm")]
        pcf = {cn: np.mean([p[cn] for p in pcs]) for cn in CLASS_NAMES} if pcs else {}
    for cn in CLASS_NAMES: det.loc[cn, iv] = pcf.get(cn, np.nan)
fig, ax = plt.subplots(figsize=(5.5, 3.8))
im = ax.imshow(det.values.astype(float), aspect="auto", vmin=0, vmax=1, cmap="viridis")
ax.set_xticks(range(4)); ax.set_xticklabels([f"{iv}s" for iv in INTERVALS])
ax.set_yticks(range(C)); ax.set_yticklabels(CLASS_NAMES)
for i in range(C):
    for j in range(4):
        v = det.values[i, j]
        if not np.isnan(v): ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                                    color="white" if v < .6 else "black", fontsize=7)
plt.colorbar(im, label="F1 par classe (LightGBM, audite)")
ax.set_title("Detectabilite par famille vs intervalle")
plt.tight_layout(); plt.savefig(FIGS / "fig3_interval_heatmap.png"); plt.savefig(FIGS / "fig3_interval_heatmap.pdf"); plt.show()

# --- Figure 5 : cout vs performance ---
if "cost" in RESULTS:
    fig, ax = plt.subplots(figsize=(5.5, 4))
    for mname, c_ in RESULTS["cost"].items():
        mf1 = agg(mname, "audited")[0]
        if np.isnan(mf1): continue
        ax.scatter(c_["throughput_512"], mf1, s=25)
        ax.annotate(mname, (c_["throughput_512"], mf1), fontsize=7,
                    xytext=(4, 3), textcoords="offset points")
    ax.set_xscale("log"); ax.set_xlabel("debit CPU (flux/s, batch 512)")
    ax.set_ylabel("macro-F1 (audite, 5 graines)")
    ax.set_title("Cout d'inference vs performance")
    plt.tight_layout(); plt.savefig(FIGS / "fig5_cost.png"); plt.savefig(FIGS / "fig5_cost.pdf"); plt.show()

# --- Figure 6 : matrice de confusion du meilleur modele ---
best = max([m for m in ALL_MODELS if m != "majority"], key=lambda m: agg(m, "audited")[0])
pte = load_probs(f"{best}|audited|strat_seed1")[1]
yte_ref = y[splits["strat_seed1"][2]]
cm = confusion_matrix(yte_ref, pte.argmax(1), labels=range(C), normalize="true")
fig, ax = plt.subplots(figsize=(5.5, 4.6))
im = ax.imshow(cm, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(C)); ax.set_xticklabels(CLASS_NAMES, rotation=45, ha="right", fontsize=7)
ax.set_yticks(range(C)); ax.set_yticklabels(CLASS_NAMES, fontsize=7)
ax.set_title(f"Matrice de confusion — {best} (audite, graine 1)")
plt.colorbar(im); plt.tight_layout()
plt.savefig(FIGS / "fig6_confusion.png"); plt.savefig(FIGS / "fig6_confusion.pdf"); plt.show()

# --- Tables LaTeX ---
def fmt(x, pct=False): return ("--" if np.isnan(x) else (f"{x*100:.2f}" if pct else f"{x:.4f}"))
rows = []
for iv in INTERVALS:
    s = RESULTS["interval_stats"][iv]
    rows.append(f"{iv}\\,s & {s['total']:,} & {s['benign']:,} & {s['benign_share']*100:.1f}\\% \\\\")
(TABS / "t1_intervals.tex").write_text("\n".join(rows))

rows = []
for mname in ALL_MODELS:
    m_, s_ = agg(mname, "audited")
    k1 = f"{mname}|audited|strat_seed1"
    r = RESULTS["models"].get(k1, {})
    fpr = r.get("binary", {}).get("fpr"); mcc = r.get("mcc", np.nan)
    rows.append(f"{mname} & {fmt(m_)}$\\pm${fmt(s_)} & {fmt(mcc)} & "
                f"{(fpr*100 if fpr is not None else np.nan):.3f}\\% \\\\")
(TABS / "t2_main.tex").write_text("\n".join(rows))

rows = []
for mname in ALL_MODELS:
    if mname == "majority": continue
    va = [agg(mname, "full")[0] if mname in FAST else one(mname, "full", "strat_seed1"),
          agg(mname, "clean")[0], agg(mname, "audited")[0], one(mname, "audited", "chrono")]
    rows.append(mname + " & " + " & ".join(fmt(v) for v in va) + " \\\\")
(TABS / "t3_before_after.tex").write_text("\n".join(rows))

if "cost" in RESULTS:
    rows = []
    for mname, c_ in RESULTS["cost"].items():
        rows.append(f"{mname} & {c_['lat_p50_ms']:.2f} & {c_['lat_p99_ms']:.2f} & "
                    f"{c_['throughput_512']:,.0f} & {c_['size_mb']:.2f} \\\\")
    (TABS / "t4_cost.tex").write_text("\n".join(rows))

save_results()
# archive figures+tables+json uniquement (les probs restent sur Drive)
tmp = pathlib.Path("/content/export"); shutil.rmtree(tmp, ignore_errors=True); tmp.mkdir()
shutil.copytree(FIGS, tmp / "figures"); shutil.copytree(TABS, tmp / "tables")
shutil.copy(RES_PATH, tmp / "article1_results.json")
shutil.make_archive("/content/article1_figures_tables", "zip", tmp)
from google.colab import files
files.download("/content/article1_figures_tables.zip")
files.download(str(RES_PATH))
print("Termine — envoyer article1_results.json et article1_figures_tables.zip")


## À me renvoyer

1. **`article1_results.json`** — tous les chiffres (c'est le fichier dont j'ai besoin pour écrire).
2. **`article1_figures_tables.zip`** — figures 300 dpi (PNG+PDF) + fragments de tables LaTeX.
3. Toute cellule en erreur avec son message.

**Si la session Colab expire en cours de route :** relancez simplement *Exécution → Tout exécuter* —
tout ce qui est déjà calculé est détecté (`runs deja calcules : N`) et sauté.

**Ordre de grandeur des durées** (GPU T4) : blocs A–B ≈ 10 min ; C ≈ 2–3 h (le trio + FT-Transformer
dominent) ; D–E ≈ 1.5–2 h ; F ≈ 20 min ; G ≈ 5 min ; H ≈ 1–1.5 h (l'intervalle 5 s est le plus gros) ;
I ≈ 15 min ; J–K ≈ 10 min.
